# Grid intraday de reação de mercado — FOMC

Gera **apenas a figura** (grid 3×3 estilo COPOM) da reação intraday ao FOMC,
exibida inline. **Não** exporta Word, **não** atualiza xlsx e **não** salva PNG —
só o gráfico na tela.

1. Configure a data e o horário da decisão (fixo em ET, convertido p/ Brasília).
2. Rode a última célula para buscar os dados no Bloomberg e desenhar o grid.

In [ ]:
# Setup + imports
import sys
from datetime import datetime
from pathlib import Path

project_root = Path.cwd().parents[3]
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from reports.fomc.core.data_loader import fetch_market_reaction, _TZ_ET, _TZ_BRT
from reports.fomc.core.word_export import create_market_reaction_grid

print("Módulos carregados.")

In [ ]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================
# Data da reunião (YYYYMMDD)
MEETING_DATE = '20260617'

# Horário da decisão — FIXO em horário de NY (ET); conversão p/ Brasília é
# automática (auto-DST via zoneinfo — hoje +1h; após 01/Nov/2026, +2h).
FOMC_DECISION_TIME_ET = "14:00:00"

# Eixo X se estende até este horário (em Brasília), mesmo sem dados até lá.
GRID_X_END_BRT = "20:00:00"

In [ ]:
# Gera e exibe o grid (sem salvar arquivo)
def _et_to_brt(meeting_date: str, hms: str) -> pd.Timestamp:
    """'HH:MM:SS' em ET (na data da reunião) -> Timestamp em Brasília."""
    md = datetime.strptime(meeting_date, "%Y%m%d").date()
    t = datetime.strptime(hms, "%H:%M:%S").time()
    return pd.Timestamp(datetime.combine(md, t), tz=_TZ_ET).tz_convert(_TZ_BRT)

def _brt_time(meeting_date: str, hms: str) -> pd.Timestamp:
    """'HH:MM:SS' já em Brasília -> Timestamp na data da reunião."""
    md = datetime.strptime(meeting_date, "%Y%m%d").date()
    t = datetime.strptime(hms, "%H:%M:%S").time()
    return pd.Timestamp(datetime.combine(md, t), tz=_TZ_BRT)

event_times = {"decisao": _et_to_brt(MEETING_DATE, FOMC_DECISION_TIME_ET)}
axis_end = _brt_time(MEETING_DATE, GRID_X_END_BRT)
print(f"Decisão FOMC: {event_times['decisao']:%H:%M} BRT  |  eixo X até {axis_end:%H:%M} BRT")

market_data = fetch_market_reaction(MEETING_DATE)
fig = create_market_reaction_grid(
    market_data,
    event_times=event_times,
    meeting_date=MEETING_DATE,
    axis_end=axis_end,
)  # sem output_path -> não salva PNG
display(fig)
plt.close(fig)  # evita exibição duplicada pelo backend inline